## What is Causal Inference?

**Causal inference** is the science of determining whether X **causes** Y, not just whether they're correlated.

### The Core Problem: Correlation ≠ Causation

**Example 1: Ice cream sales and drowning deaths**

Observation: Cities with higher ice cream sales have more drowning deaths.

❌ **Correlation thinking**: "Ice cream causes drowning! Ban ice cream!"  
✅ **Causal thinking**: Both are caused by hot weather (confounding variable).

```
Hot weather → More people swim → More drownings
Hot weather → More ice cream sales
```

### Why This Matters at ServiceTitan

ServiceTitan makes thousands of product decisions based on data. Without causal inference, you ship features that don't work:

| Question | Correlation Answer | Causal Answer |
|----------|-------------------|---------------|
| Does Atlas increase revenue? | Contractors using Atlas have 20% higher revenue | Maybe — but did Atlas **cause** that, or do high-performing contractors adopt tools earlier? |
| Should we train technicians on upselling? | Techs with high upsell rates have higher job values | Maybe — but are those techs naturally better salespeople, or did training help? |
| Does faster dispatch increase job count? | Faster-dispatching contractors book more jobs | Maybe — but do busy contractors dispatch faster **because** they're busy? |

**The danger**: Ship "Atlas works!" based on correlation → Atlas doesn't actually help → wasted engineering years.

### The Fundamental Problem of Causal Inference

You can never observe both outcomes for the same unit at the same time:

**Example: Does Atlas increase a contractor's monthly revenue?**

```
Contractor #123 in March 2025:

Reality (what we observe):
  ✅ Uses Atlas     → Revenue = $150,000

Counterfactual (what we can't observe):
  ❓ Doesn't use Atlas → Revenue = ???
```

We **can't go back in time** and see what would have happened without Atlas.

### The Solution: Causal Inference Methods

Since we can't observe counterfactuals for individuals, we use statistical techniques to estimate causal effects:

---

#### 1. Randomized A/B Test (The Gold Standard)

**What it is:**  
Randomly assign users to treatment (new feature) or control (old feature), then compare outcomes.

**Why it works:**  
Random assignment ensures the two groups are **identical on average** in all ways except the treatment. Any difference in outcomes must be caused by the treatment.

**Example at ServiceTitan:**
```python
# Randomly assign contractors to Atlas vs no-Atlas
treatment = contractors.sample(frac=0.5, random_state=42)
control = contractors[~contractors.index.isin(treatment.index)]

# After 1 month, compare revenue
treatment_revenue = treatment['revenue'].mean()  # $152,000
control_revenue = control['revenue'].mean()      # $150,000

causal_effect = treatment_revenue - control_revenue  # $2,000
```

**When to use:** Whenever you can control who gets the treatment (most product features)

**Limitation:** Can't use if you can't isolate users (e.g., dispatcher tools affect whole teams)

---

#### 2. CUPED (Controlled-experiment Using Pre-Experiment Data)

**What it is:**  
An enhancement to A/B tests that reduces variance by using pre-treatment data as a covariate.

**Why it works:**  
If you measure the outcome variable **before** the experiment, you can "subtract out" the noise that comes from users being naturally different. This lets you detect smaller effects with fewer users.

**The formula:**
```
adjusted_outcome = actual_outcome - θ * (pre_treatment_value - mean_pre_treatment)
```

Where θ is chosen to maximize variance reduction (it's the correlation coefficient).

**Example at ServiceTitan:**
```python
# Without CUPED: need 2,000 contractors to detect $50 lift
# With CUPED: need 500 contractors (4x faster!)

# Why? Revenue-per-job has σ = $400 (high variance)
# But if we use "last month's revenue" as a covariate,
# we can explain away 75% of the variance, reducing σ_effective to $200
```

**When to use:** A/B tests where the metric is noisy (revenue, retention) and you have historical data

**Limitation:** Requires pre-treatment measurements of the outcome

---

#### 3. Switchback Experiment

**What it is:**  
Alternate between treatment and control over **time periods** (e.g., weeks) rather than assigning individual users.

**Why you need it:**  
Some features can't be isolated to individual users. Example: A new dispatcher tool changes how the **entire dispatch team** works. You can't give it to half the dispatchers without affecting the other half.

**How it works:**
```
Week 1: Everyone uses OLD tool → measure jobs dispatched
Week 2: Everyone uses NEW tool → measure jobs dispatched
Week 3: OLD tool
Week 4: NEW tool
...
Compare average performance in NEW weeks vs OLD weeks
```

**Example at ServiceTitan:**  
Testing a new dispatch algorithm that re-orders the job queue for all dispatchers.

**When to use:** Network effects, shared resources, team-level features

**Limitation:** Need to control for time trends (busy season affects both treatment/control weeks)

---

#### 4. Difference-in-Differences (DiD)

**What it is:**  
Compare the **change over time** in a treated group vs an untreated group.

**Why you need it:**  
Sometimes you can't randomize (e.g., rolled out Atlas to West Coast first for logistical reasons). But you can compare how West Coast changed vs East Coast (which didn't get Atlas yet).

**The formula:**
```
Effect = (Treatment_after - Treatment_before) - (Control_after - Control_before)
         └──────────── change in treated ──────┘   └───────── change in control ──────┘
```

**Example at ServiceTitan:**
```
Atlas rollout to West Coast in July 2024:

West Coast revenue:
  June (before):  $140K/contractor
  August (after): $155K/contractor  → +$15K change

East Coast revenue (no Atlas yet):
  June:  $138K/contractor
  August: $148K/contractor  → +$10K change

DiD estimate = $15K - $10K = $5K causal effect
                              ↑
                    (The $10K is general market growth,
                     the extra $5K is from Atlas)
```

**Parallel trends assumption:** Treatment and control would have followed the same trend without treatment

**When to use:** Staged rollouts, can't randomize, have pre/post data for both groups

**Limitation:** Requires parallel trends (test this with pre-treatment data)

---

#### 5. Propensity Score Matching (PSM)

**What it is:**  
Match each treated user with a similar untreated user based on **all observed characteristics**, then compare outcomes.

**Why you need it:**  
Last resort when you can't randomize and don't have good pre/post periods. Example: Contractors self-select into Atlas. High-performers adopted it first. You want to know: "What would a typical Atlas user's revenue be **if they hadn't adopted Atlas**?"

**How it works:**
1. Build a model predicting probability of treatment: `P(uses_Atlas | size, region, revenue_history, ...)`
2. This probability is the "propensity score"
3. Match each treated user with an untreated user who has a similar propensity score
4. Compare outcomes within matched pairs

**Example at ServiceTitan:**
```python
# Contractor A: uses Atlas, propensity score = 0.72
# Find contractors who DON'T use Atlas with similar score:
#   Contractor B: no Atlas, propensity = 0.71  ← good match!
#   Contractor C: no Atlas, propensity = 0.73  ← good match!

# Compare A's revenue to (B+C)/2
```

**Why it works:** By matching on propensity, you simulate what randomization would have done.

**When to use:** Observational data only, no randomization, no good time-series structure

**Limitation:** Only controls for **observed** confounders. If there's an unobserved confounder (e.g., "motivation"), PSM can't fix it. **Always prefer A/B tests.**

---

### Summary Table

| Method | When to Use | Key Advantage | Key Limitation |
|--------|-------------|---------------|----------------|
| **Randomized A/B test** | You control assignment | Gold standard — eliminates all confounding | Can't use when treatment affects others (network effects) |
| **CUPED** | A/B test with noisy metrics | Cuts sample size by 2-4x | Requires pre-treatment data |
| **Switchback** | Can't isolate individuals (e.g., dispatcher tools) | Handles network effects | Must control for time trends |
| **Difference-in-differences** | Rolled out to some regions first | Works with staged rollouts | Requires parallel trends assumption |
| **Propensity score matching** | Can't randomize, only observe | Last resort for observational data | Can't control for unobserved confounders |

**Decision tree:**
1. Can you randomize? → **A/B test** (+ CUPED if metrics are noisy)
2. Can't isolate users? → **Switchback**
3. Staged rollout? → **Difference-in-differences**
4. Observational only? → **Propensity score matching** (weakest, but better than nothing)


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(0)


---
## Section 1 — Power and Sample Size

Before running any experiment, compute the sample size needed to detect your
target effect with adequate power. The four knobs:

| Knob | Typical value | Notes |
|------|---------------|-------|
| α (false positive rate) | 0.05 | Two-sided |
| β (false negative rate) | 0.20 (= 80% power) | Industry standard |
| MDE (minimum detectable effect) | depends — set by product team | THE business question |
| σ (outcome std-dev) | measured from history | The hardest part |

### Sample size formula (two-sample mean test)

$$
n = \frac{2 (z_{\alpha/2} + z_\beta)^2 \sigma^2}{\text{MDE}^2}
$$

per arm. So total N is 2n. The key insight: **n grows quadratically with σ
and inversely with MDE²**. Cutting MDE in half quadruples your sample size.

For ServiceTitan, σ on revenue-per-job is huge relative to mean (revenue is
heavy-tailed), so even a 5% lift requires thousands of jobs per arm. **This is
why CUPED matters** (Section 3).


In [ ]:
def sample_size_two_means(mde: float, sigma: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """Sample size per arm for two-sample mean test. Returns n per arm (so total = 2n)."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    n_per_arm = 2 * (z_alpha + z_beta) ** 2 * sigma**2 / mde**2
    return int(np.ceil(n_per_arm))


def sample_size_proportions(p_baseline: float, mde_abs: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """Sample size per arm for difference-in-proportions test."""
    p1, p2 = p_baseline, p_baseline + mde_abs
    pooled = (p1 + p2) / 2
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    n_per_arm = ((z_alpha * np.sqrt(2 * pooled * (1 - pooled)) +
                  z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2)))**2) / mde_abs**2
    return int(np.ceil(n_per_arm))


# ── ST-flavored examples ─────────────────────────────────────────────────
print("Example 1: Revenue-per-job, want to detect $50 lift")
print(f"  σ = $400 (revenue/job std-dev)")
n = sample_size_two_means(mde=50, sigma=400)
print(f"  n per arm: {n:,}   total: {2*n:,}\n")

print("Example 2: Same lift, σ down to $200 (e.g., after CUPED)")
n = sample_size_two_means(mde=50, sigma=200)
print(f"  n per arm: {n:,}   total: {2*n:,}   ← 4x fewer samples\n")

print("Example 3: Atlas conversion rate, baseline 12%, want to detect 1pp lift")
n = sample_size_proportions(p_baseline=0.12, mde_abs=0.01)
print(f"  n per arm: {n:,}   total: {2*n:,}")


---
## Section 2 — Why Naive A/B Testing Fails

Three failure modes that ship a "winner" that isn't.

### 2a. Peeking inflates false positives

If you check the test daily and ship when `p < 0.05` first crosses, your true
false positive rate is much higher than 5% — closer to 20-30%. The math: every
peek is another chance to randomly cross the threshold under the null.

### 2b. Multiple comparisons

If you run 20 metrics and bonferroni-correct nothing, on average **one will
hit p < 0.05 even when no effect exists**.

### 2c. Heterogeneous treatment effects masquerading as null

Average effect can be zero while two segments have offsetting effects.


In [ ]:
# ── 2a: Peeking simulation ────────────────────────────────────────────────
def peeking_false_positive_rate(n_total: int = 5_000, n_peeks: int = 50, n_runs: int = 500):
    """Run many simulated A/A tests, check how often peeking-then-stopping declares 'significant'."""
    rng = np.random.default_rng(0)
    false_positives = 0
    for _ in range(n_runs):
        # A/A: both arms drawn from same distribution
        a = rng.normal(100, 30, n_total)
        b = rng.normal(100, 30, n_total)
        # peek at evenly-spaced sample sizes
        peek_at = np.linspace(100, n_total, n_peeks).astype(int)
        for k in peek_at:
            t, p = stats.ttest_ind(a[:k], b[:k])
            if p < 0.05:
                false_positives += 1
                break  # would have stopped here in practice
    return false_positives / n_runs


fpr = peeking_false_positive_rate(n_total=5_000, n_peeks=50, n_runs=500)
print(f"Peeking 50 times in an A/A test:")
print(f"  Expected FPR (no peeking): 5.0%")
print(f"  Actual FPR (with peeking): {fpr:.1%}   ← inflated")
print(f"\n  Fix: pre-register a single analysis time, OR use sequential testing (e.g., O'Brien-Fleming bounds).")


In [ ]:
# ── 2b: Multiple comparisons ──────────────────────────────────────────────
def multiple_comparisons_demo(n_metrics: int = 20, n_runs: int = 500):
    rng = np.random.default_rng(0)
    any_significant = 0
    for _ in range(n_runs):
        # Simulate 20 metrics under the null (no effect on any)
        for _ in range(n_metrics):
            a = rng.normal(0, 1, 500)
            b = rng.normal(0, 1, 500)
            _, p = stats.ttest_ind(a, b)
            if p < 0.05:
                any_significant += 1
                break
    return any_significant / n_runs


rate = multiple_comparisons_demo(n_metrics=20)
print(f"Probability of ANY metric crossing p<0.05 across 20 null tests: {rate:.1%}")
print(f"  Theoretical: 1 - 0.95^20 = {1 - 0.95**20:.1%}")
print(f"\n  Fix: Bonferroni (α/k), Benjamini-Hochberg FDR control, or pre-register a primary metric.")


In [ ]:
# ── 2c: Heterogeneous effects masking null ────────────────────────────────
rng = np.random.default_rng(0)

# Two segments: VIP customers (small) and standard (large)
# Treatment helps VIPs (+$30) but hurts standards (-$5).
# Average effect is small, but it's not zero — and it's not the same direction.
n_vip = 200
n_std = 1800
revenue_vip_control   = rng.normal(500, 100, n_vip)
revenue_vip_treatment = rng.normal(530, 100, n_vip)    # +30 lift
revenue_std_control   = rng.normal(150, 50,  n_std)
revenue_std_treatment = rng.normal(145, 50,  n_std)    # -5 hit

control = np.concatenate([revenue_vip_control, revenue_std_control])
treatment = np.concatenate([revenue_vip_treatment, revenue_std_treatment])

t_overall, p_overall = stats.ttest_ind(treatment, control)
t_vip, p_vip = stats.ttest_ind(revenue_vip_treatment, revenue_vip_control)
t_std, p_std = stats.ttest_ind(revenue_std_treatment, revenue_std_control)

print("Heterogeneous effects:")
print(f"  Overall    Δ=${treatment.mean()-control.mean():+.2f}  p={p_overall:.3f}")
print(f"  VIP       Δ=${revenue_vip_treatment.mean()-revenue_vip_control.mean():+.2f}  p={p_vip:.3f}")
print(f"  Standard  Δ=${revenue_std_treatment.mean()-revenue_std_control.mean():+.2f}  p={p_std:.3f}")
print(f"\n  Lesson: always check pre-registered subgroup analyses, not just headline metric.")


---
## Section 3 — CUPED Variance Reduction

**CUPED** (Controlled-experiment Using Pre-Experiment Data, Microsoft 2013) is
the most-used variance reduction trick in industry A/B testing. The intuition:
much of the variance in your outcome metric is **driven by user-level baselines
that exist before treatment**. Subtract that pre-treatment signal and the
residual variance drops 30-60%.

### The formula

```
y_cuped = y - θ (x - x̄)

where θ = Cov(y, x) / Var(x)
      x = pre-treatment value of the metric (or related metric)
      y = observed outcome
```

`y_cuped` has the same mean as `y` (so the treatment effect is unchanged) but
**lower variance**, so the t-statistic is larger and you reach significance
faster. This is why σ shrinks in the Section 1 examples — CUPED is *the*
mechanism for that shrink.

### When to use it

Whenever you have pre-period data for the same units. Almost always at ST:
contractors have months of history before any experiment. Skip it for new-user
metrics where pre-period is undefined.


In [ ]:
def apply_cuped(y_post: np.ndarray, x_pre: np.ndarray):
    """Return CUPED-adjusted y, theta, and variance reduction percentage."""
    x_centered = x_pre - x_pre.mean()
    theta = np.cov(y_post, x_pre, ddof=1)[0, 1] / np.var(x_pre, ddof=1)
    y_adj = y_post - theta * x_centered
    var_reduction = 1 - np.var(y_adj, ddof=1) / np.var(y_post, ddof=1)
    return y_adj, theta, var_reduction


# ── Simulate experiment with strong user-level pre-period correlation ──
rng = np.random.default_rng(1)
N = 2000
user_baseline = rng.normal(100, 30, N)      # pre-period mean revenue
treatment_mask = rng.random(N) < 0.5
treatment_effect_true = 5.0
y_post = user_baseline + rng.normal(0, 10, N) + treatment_effect_true * treatment_mask

x_pre = user_baseline + rng.normal(0, 5, N)  # noisy observation of baseline

# Naive comparison
t_naive, p_naive = stats.ttest_ind(y_post[treatment_mask], y_post[~treatment_mask])
delta_naive = y_post[treatment_mask].mean() - y_post[~treatment_mask].mean()

# CUPED-adjusted
y_adj, theta, var_red = apply_cuped(y_post, x_pre)
t_cuped, p_cuped = stats.ttest_ind(y_adj[treatment_mask], y_adj[~treatment_mask])
delta_cuped = y_adj[treatment_mask].mean() - y_adj[~treatment_mask].mean()

print(f"True treatment effect: +${treatment_effect_true:.2f}\n")
print(f"NAIVE:")
print(f"  Estimated effect: +${delta_naive:.2f}")
print(f"  t = {t_naive:.2f},  p = {p_naive:.4f}\n")
print(f"CUPED (θ={theta:.3f}, variance reduction = {var_red:.1%}):")
print(f"  Estimated effect: +${delta_cuped:.2f}")
print(f"  t = {t_cuped:.2f},  p = {p_cuped:.4f}")
print(f"\n  Same point estimate, much larger t-stat → reach significance with fewer samples.")


---
## Section 4 — Switchback Experiments

Some changes can't be randomized at the user level because the experimental
unit isn't isolatable. Classic example: **dispatcher logic**. If you A/B test
two dispatch algorithms on the same dispatcher, the techs from arm A are
unavailable for arm B's jobs, contaminating the experiment.

**Switchback** randomizes by **time slice** instead of by user:

```
Hour 0-1:  arm A
Hour 1-2:  arm B
Hour 2-3:  arm A
...
```

Each slice is a unit. Then you fit:

$$y_t = \alpha + \tau \cdot \mathbb{1}[\text{arm}=B] + \gamma \cdot \text{covariates}_t + \epsilon_t$$

The estimate of τ is your treatment effect. Use Newey-West standard errors to
handle autocorrelation between adjacent slices.


In [ ]:
# ── Simulate dispatcher switchback ────────────────────────────────────────
rng = np.random.default_rng(2)
n_slices = 200  # ~8 days of hourly slices
arms = rng.choice(["A", "B"], size=n_slices)

# Confounders: time-of-day demand, day-of-week
hour = np.arange(n_slices) % 24
demand = 1.0 + 0.3 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 0.1, n_slices)

# Outcome: jobs completed per hour. True effect of arm B: +0.5 jobs/hr.
true_effect = 0.5
jobs_completed = (
    10 * demand                       # baseline volume scales with demand
    + true_effect * (arms == "B")     # arm B uplift
    + rng.normal(0, 2, n_slices)
)

df = pd.DataFrame({"arm": arms, "demand": demand, "jobs": jobs_completed, "hour": hour})

# ── Naive: ignore covariates ──────────────────────────────────────────────
mean_A = df[df["arm"] == "A"]["jobs"].mean()
mean_B = df[df["arm"] == "B"]["jobs"].mean()
t_naive, p_naive = stats.ttest_ind(df[df["arm"] == "B"]["jobs"], df[df["arm"] == "A"]["jobs"])
print(f"Naive (no covariate adjustment):")
print(f"  Δ = {mean_B - mean_A:+.3f}  jobs/hr")
print(f"  p = {p_naive:.3f}\n")

# ── Regression with covariate adjustment ──────────────────────────────────
from sklearn.linear_model import LinearRegression
X = np.column_stack([
    (df["arm"] == "B").astype(int),
    df["demand"].values,
    np.sin(2*np.pi*df["hour"]/24),
    np.cos(2*np.pi*df["hour"]/24),
])
y = df["jobs"].values
model = LinearRegression().fit(X, y)
coef_arm_B = model.coef_[0]
# Manual SE estimate
y_pred = model.predict(X)
residuals = y - y_pred
rss = (residuals**2).sum()
n, k = X.shape
sigma_hat = np.sqrt(rss / (n - k - 1))
XtX_inv = np.linalg.inv(X.T @ X)
se_arm_B = sigma_hat * np.sqrt(XtX_inv[0, 0])
t_stat = coef_arm_B / se_arm_B
p_val = 2 * (1 - stats.norm.cdf(abs(t_stat)))

print(f"Regression with covariates (demand, hour cyclic):")
print(f"  Effect of arm B: {coef_arm_B:+.3f}  jobs/hr")
print(f"  SE: {se_arm_B:.3f},  t = {t_stat:.2f},  p = {p_val:.4f}")
print(f"\n  True effect was +{true_effect}. Naive estimate is biased by demand timing")
print(f"  (random arm assignment correlated with high/low demand hours by chance).")


---
## Section 5 — Difference-in-Differences (DiD)

When you can't randomize but you do have:
- A treatment group and a comparison group, AND
- Pre/post measurements on both

DiD compares the **change** in the treatment group to the **change** in the
control group. This nets out anything that affected both groups equally.

### Atlas rollout example
Atlas is rolled out to all Pro-plan contractors in Q3 2026. Basic-plan
contractors don't get it. We want to estimate Atlas's effect on jobs-per-day.

Direct comparison Pro vs. Basic is confounded (Pros differ for many reasons).
But:

$$
\hat{\tau}_{DiD} = (\bar{Y}_{Pro,post} - \bar{Y}_{Pro,pre}) -
                    (\bar{Y}_{Basic,post} - \bar{Y}_{Basic,pre})
$$

The Basic group's change captures everything that happened in Q3 *other than
Atlas* (seasonality, the economy, etc). Subtracting it isolates Atlas's effect.

### Critical assumption: parallel trends
The two groups would have moved together absent treatment. Test by plotting
their pre-period trends — if they diverge, DiD is biased.


In [ ]:
rng = np.random.default_rng(3)
n_pro, n_basic = 200, 800
n_weeks_pre, n_weeks_post = 12, 12
true_atlas_effect = 1.2   # jobs/day uplift for Pros from Atlas

# Pro contractors: higher baseline, slightly faster growth (selection effect)
pro_pre  = rng.normal(15, 3, (n_pro, n_weeks_pre))
pro_post = rng.normal(15.5, 3, (n_pro, n_weeks_post)) + true_atlas_effect
basic_pre  = rng.normal(8, 2, (n_basic, n_weeks_pre))
basic_post = rng.normal(8.5, 2, (n_basic, n_weeks_post))    # no Atlas

# Naive: just compare Pro post vs Pro pre
delta_pro    = pro_post.mean() - pro_pre.mean()
delta_basic  = basic_post.mean() - basic_pre.mean()
did_estimate = delta_pro - delta_basic

print(f"True Atlas effect:   +{true_atlas_effect:.2f}  jobs/day")
print(f"Pro post - pre:      +{delta_pro:.2f}  ← contaminated by secular trends")
print(f"Basic post - pre:    +{delta_basic:.2f}  ← captures non-Atlas trends")
print(f"DiD estimate:        +{did_estimate:.2f}  ← isolates Atlas effect")
print(f"\nDiD reduces bias when control group's trend correctly predicts what would have happened.")

# Parallel trends check: plot weekly means pre-period
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(n_weeks_pre), pro_pre.mean(axis=0), "b-o", label="Pro (pre)")
ax.plot(range(n_weeks_pre), basic_pre.mean(axis=0), "g-s", label="Basic (pre)")
ax.set_xlabel("Week of pre-period")
ax.set_ylabel("Mean jobs/day")
ax.set_title("Parallel trends check — pre-treatment period")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/parallel_trends.png", dpi=80)
plt.close()
print("\nSaved parallel-trends chart to /tmp/parallel_trends.png")


---
## Section 6 — Propensity Score Matching

When you have *only* observational data — no pre/post split, no parallel
control group — match treated units to "twin" control units based on their
**propensity** to be treated, then compare outcomes within matched pairs.

### Setup
1. Fit a model `P(treated=1 | covariates)` — usually logistic regression
2. For each treated unit, find a control unit with similar propensity score
3. Compare outcomes within the matched set

### When this works
- Covariates capture all confounding (the "ignorability" assumption)
- Treated and control groups overlap in covariate space ("common support")

### When it fails
- Unobserved confounders (the boss-favored-the-techs-they-trained problem)
- No common support — e.g., treated units are all VIPs, no control VIPs exist


In [ ]:
# Simulate: contractors choose to enable a new feature (self-selected).
# Big shops adopt early, small shops late. Outcome is revenue growth.
# Naive comparison overstates effect because adopters were already growing faster.

rng = np.random.default_rng(4)
N = 3000
shop_size = rng.exponential(5, N)              # confounder: 1-50ish
years_in_business = rng.uniform(1, 25, N)
historical_growth = 0.05 + 0.005 * shop_size + rng.normal(0, 0.02, N)

# Probability of adopting feature increases with size + years
logit = -2 + 0.3 * shop_size + 0.05 * years_in_business
prob_adopt = 1 / (1 + np.exp(-logit))
treated = rng.random(N) < prob_adopt

# True causal effect of feature: +1% growth
true_effect = 0.01
outcome = historical_growth + true_effect * treated + rng.normal(0, 0.01, N)

df = pd.DataFrame({
    "shop_size": shop_size,
    "years": years_in_business,
    "treated": treated.astype(int),
    "outcome": outcome,
})

naive_diff = df[df["treated"] == 1]["outcome"].mean() - df[df["treated"] == 0]["outcome"].mean()
print(f"True causal effect:   +{true_effect:.4f}")
print(f"Naive difference:     +{naive_diff:.4f}  ← biased upward by selection")

# ── Fit propensity model ─────────────────────────────────────────────────
X = df[["shop_size", "years"]].values
y = df["treated"].values
ps_model = LogisticRegression().fit(X, y)
df["propensity"] = ps_model.predict_proba(X)[:, 1]

# ── Greedy nearest-neighbor matching (1:1) on propensity ─────────────────
treated_idx = df[df["treated"] == 1].index.tolist()
control_idx = df[df["treated"] == 0].index.tolist()
control_ps = df.loc[control_idx, "propensity"].values
control_used = np.zeros(len(control_idx), dtype=bool)

matched_pairs = []
for ti in treated_idx:
    p = df.at[ti, "propensity"]
    dists = np.where(control_used, np.inf, np.abs(control_ps - p))
    best = int(np.argmin(dists))
    if dists[best] < 0.05:   # caliper
        matched_pairs.append((ti, control_idx[best]))
        control_used[best] = True

matched_treated = [t for t, _ in matched_pairs]
matched_control = [c for _, c in matched_pairs]
psm_diff = df.loc[matched_treated, "outcome"].mean() - df.loc[matched_control, "outcome"].mean()

print(f"PSM estimate (n={len(matched_pairs)} pairs): +{psm_diff:.4f}  ← much closer to truth")
print(f"\nThe bias-correction is the value of PSM. It can't fix unobserved confounding —")
print(f"if there's a variable you didn't measure that drives adoption AND outcome, you're stuck.")


---
## Putting It Together — A Decision Tree for ST Experiments

When the product team asks "did this thing work?", walk this tree:

```
Can you randomize?
├── YES, at user level
│   └── Standard A/B test + CUPED if you have pre-period data
├── YES, but only at time-slice level (dispatcher, prices)
│   └── Switchback + regression with covariate adjustment
├── NO, but you have a comparable group that didn't get treated
│   └── Difference-in-differences (check parallel trends first!)
└── NO, and you only have observational data
    └── Propensity score matching (and pray your covariates are sufficient)
```

And **regardless of method**, before launching:
- Pre-register the analysis (metric, sample size, stopping rule, subgroups)
- Run an A/A test first to verify your pipeline doesn't have a bug
- Set up sequential testing bounds if you must peek

These together cover ~95% of the experimental decisions ST DS teams make.


In [ ]:
print("=" * 60)
print("Notebook 25 — Causal Inference & Experimental Design")
print("=" * 60)
print(f"  Section 1 — power & sample size:  3 worked examples")
print(f"  Section 2 — failure modes:        peeking, multiple comparisons, heterogeneity")
print(f"  Section 3 — CUPED:                applied, variance reduction quantified")
print(f"  Section 4 — switchback:           regression w/ covariate adjustment")
print(f"  Section 5 — DiD:                  Atlas rollout simulation")
print(f"  Section 6 — propensity matching:  bias-correction demo")
print("=" * 60)
